# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a website data,  
and responds with improvements suggestions can be done.

In [ ]:
import asyncio
import sys
import openai
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

class WebsiteAnalyzer:
    def __init__(self, url):
        self.url = url
        self.title = ""
        self.text = ""
        self.suggestions = ""

    async def initialize(self):
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            context = await browser.new_context(
                user_agent='Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36',
                viewport={'width': 1920, 'height': 1080},
            )
            page = await context.new_page()
            try:
                await page.goto(self.url, timeout=90000)
                await page.wait_for_load_state('networkidle', timeout=30000)
                self.title = await page.title()
                content = await page.content()
                soup = BeautifulSoup(content, 'html.parser')
                for tag in soup.find_all(["script", "style", "img", "input"]):
                    tag.decompose()
                self.text = soup.body.get_text(separator="\n", strip=True) if soup.body else ""
                self.suggestions = self.get_suggestions()
            finally:
                await browser.close()

    def get_suggestions(self):
        response = openai.chat.completions.create(
            model="llama3.2:1b",
            messages=[{"role": "user", "content": f"Analyze this website and give UX suggestions.\nTitle: {self.title}\nContent: {self.text[:4000]}"}]
        )
        return response.choices[0].message.content

async def main():
    url = "https://medium.com/@copy.kolary/10-simple-and-effective-ways-to-develop-creativity-65faf94bd442"
    analyzer = WebsiteAnalyzer(url)
    await analyzer.initialize()
    print("\nTitle:\n", analyzer.title)
    print("\nSuggestions:\n", analyzer.suggestions)

if __name__ == "__main__":
    asyncio.run(main())